In [ ]:
from prediction import prediction

import os
import pandas as pd

# List of target teams
target = ["BOS", "MEM", "CHI", "OKC", "BKN", "DAL", "HOU", "LAL"]
df_list = []  # List to store DataFrames

# Loop through each team's roster file
for team in target:
    roster_path = f"D:\\roster_folder\\2024\\{team}_roster_file.csv"  # Construct the file path

    if os.path.exists(roster_path):  # Check if file exists
        df = pd.read_csv(roster_path)  # Read CSV file
        df_list.append(df)  # Store DataFrame in list
    else:
        print(f"File not found: {roster_path}")  # Debugging message

# Combine all DataFrames if any were found
if df_list:
    df_combined = pd.concat(df_list, ignore_index=True)

    # Extract two columns as a dictionary
    player_dict = dict(zip(df_combined['PLAYER_uni'], df_combined['api_team_name']))

    print(player_dict)  # Display dictionary
else:
    print("No valid roster files found.")





# remember to run mover
# remember to check schedule csv
import pandas as pd
date_list = ["2023-24","2024-25"]
stats_path = {
    'usage_path':'D:/nba_usage_csv_historic/usage_csv_{date}/{date}_content.csv',
    'catch_shoot':"D:/nba_tracking_data_csv/nba_csv_{date}/catch_shoot_content.csv",
    'drives':"D:/nba_tracking_data_csv/nba_csv_{date}/drives_content.csv",
    'elbow_touches':"D:/nba_tracking_data_csv/nba_csv_{date}/elbow_touch_content.csv",
    'paint_touches':"D:/nba_tracking_data_csv/nba_csv_{date}/paint_touch_content.csv",
    'passing':"D:/nba_tracking_data_csv/nba_csv_{date}/passing_content.csv",
    'pullup':"D:/nba_tracking_data_csv/nba_csv_{date}/pullup_content.csv",
    'shooting_efficiency':"D:/nba_tracking_data_csv/nba_csv_{date}/shooting_efficiency_content.csv",
    'touches':"D:/nba_tracking_data_csv/nba_csv_{date}/touches_content.csv",
    'tracking_post_ups_content':"D:/nba_tracking_data_csv/nba_csv_{date}/tracking_post_ups_content.csv"
}
schedule_base_path = "D:/nba_scheduled_csv/schedule_csv_2025/{schedule_team}_schedule_content.csv"
player_base_path = "D:/nba_player_csv_historic/season_{date}/all_quarters/{player}_content.csv"
defense_base_path = "D:/nba_defense_history_csv/defense_csv_{date}/all_quarter_defense_content.csv"

results_reb = prediction(player_dict, date_list, stats_path, player_base_path, defense_base_path, schedule_base_path,'REB','REB')
results_ast = prediction(player_dict, date_list, stats_path, player_base_path, defense_base_path, schedule_base_path,'AST','AST')
results_pts = prediction(player_dict, date_list, stats_path, player_base_path, defense_base_path, schedule_base_path,'PTS','PTS')
results_3pm = prediction(player_dict, date_list, stats_path, player_base_path, defense_base_path, schedule_base_path,'3PM','3PM')
results_pts = results_pts.rename(columns={'FGA': 'PTS'})



# display(results_reb)
# display(results_ast)
# display(results_pts)
# display(results_3pm)








In [ ]:

display(results_reb)
display(results_ast)
display(results_pts)
display(results_3pm)

results_3pm = results_3pm[results_3pm["confidence_level_3PM"] != 100.00]
df_merged = results_reb.merge(results_ast, on=['Player', 'team'], suffixes=('_reb', '_ast')) \
    .merge(results_pts, on=['Player', 'team'], suffixes=('_pts','extra_pts')) \
    .merge(results_3pm, on=['Player', 'team'], suffixes=('_pts','_3pm'))


players_to_remove = [
    # "Jordan Poole",
    # "Bub Carrington",
    # "A.J. Johnson",
    # "Justin Champagnie",
    # "Alex Sarr",
    # "Quentin Grimes",
    # "Justin Edwards",
    # "Jared Butler",
    # "Ricky Council IV",
    # "Guerschon Yabusele",
    # "Austin Reaves",
    # "LeBron James",
    # "Luka Doncic",
    # "Dorian Finney-Smith",
    # "Jaxson Hayes",
    # "Andrew Nembhard",
    # "Aaron Nesmith",
    # "Tyrese Haliburton",
    # "Pascal Siakam",
    # "Myles Turner",
    # "Kris Dunn",
    # "Norman Powell",
    # "James Harden",
    # "Kawhi Leonard",
    # "Ivica Zubac",
    # "Mikal Bridges",
    # "Josh Hart",
    # "Cameron Payne",
    # "OG Anunoby",
    # "Karl-Anthony Towns",
    # "RJ Barrett",
    # "Ochai Agbaji",
    # "Jamal Shead",
    # "Scottie Barnes",
    # "Orlando Robinson",
    # "Keon Johnson",
    # "Ziaire Williams",
    # "D'Angelo Russell",
    # "Jalen Wilson",
    # "Nic Claxton",
    # "Gary Trent Jr.",
    # "Taurean Prince",
    # "Ryan Rollins",
    # "Kyle Kuzma",
    # "Brook Lopez",
    # "Christian Braun",
    # "Michael Porter Jr.",
    # "Jamal Murray",
    # "Aaron Gordon",
    # "Nikola Jokic",
    # "Jrue Holiday",
    # "Jaylen Brown",
    # "Derrick White",
    # "Jayson Tatum",
    # "Kristaps Porzingis"
    # "Devin Booker",
    # "Ryan Dunn",
    # "Collin Gillespie",
    # "Kevin Durant",
    # "Nick Richards"
    ]

# players_to_remove = ['Tyrese Haliburton', 'Player1', 'Player2', 'Player3']

df_merged = df_merged[~df_merged["Player"].isin(players_to_remove)]



from parlay_picker import create_parlays

num_groups = 10  # Number of parlays you want to generate
players_per_group = 4 # Number of players in each parlay
min_confidence = 60
threepm_confidence = 50

parlays_df, group_confidence_scores = create_parlays(df_merged, num_groups, players_per_group,min_confidence,threepm_confidence)


# Display the confidence scores for each group
print("Confidence scores for each group:")
for group, score in group_confidence_scores:
    print(f"{group}: {score:.2f}")


# Display the DataFrames for each parlay and their confidence scores
for (parlay_name, score), parlay_df in zip(group_confidence_scores, parlays_df):
    print("\n")
    print(f"{parlay_name}: total confidence score: {score:.2f}")
    display(parlay_df)
    print("\n")


In [ ]:
def get_safe_bets(df, confidence_threshold=60):
    """
    Filters the DataFrame for safe bets based on confidence threshold.
    Selects the lowest value in the stat range for a conservative bet.
    """
    safe_bets = df[df["confidence_level_PTS"] >= confidence_threshold].copy()
    
    # Extract the lowest stat value from the "Stat Range" column
    safe_bets["Safe Bet"] = safe_bets["PTS"].apply(lambda x: int(x.split(" - ")[0]))
    
    # Select only necessary columns
    safe_bets = safe_bets[["Player", "team", "PTS", "Safe Bet", "confidence_level_PTS"]]
    
    return safe_bets

# Example Usage:
safe_bets_df = get_safe_bets(results_pts, confidence_threshold=60)
print(safe_bets_df)

In [ ]:
def get_safe_bets(df, confidence_threshold=60):
    """
    Filters the DataFrame for safe bets based on confidence threshold.
    Selects the lowest value in the stat range for a conservative bet.
    """
    safe_bets = df[df["confidence_level_REB"] >= confidence_threshold].copy()
    
    # Extract the lowest stat value from the "Stat Range" column
    safe_bets["Safe Bet"] = safe_bets["REB"].apply(lambda x: int(x.split(" - ")[0]))
    
    # Select only necessary columns
    safe_bets = safe_bets[["Player", "team", "REB", "Safe Bet", "confidence_level_REB"]]
    
    return safe_bets

# Example Usage:
safe_bets_df = get_safe_bets(results_reb, confidence_threshold=60)
print(safe_bets_df)

In [ ]:
def get_safe_bets(df,target ,confidence_threshold=60):
    """
    Filters the DataFrame for safe bets based on confidence threshold.
    Selects the lowest value in the stat range for a conservative bet.
    """
    safe_bets = df[df[f"confidence_level_{target}"] >= confidence_threshold].copy()
    
    # Extract the lowest stat value from the "Stat Range" column
    safe_bets["Safe Bet"] = safe_bets[target].apply(lambda x: int(x.split(" - ")[0]))
    
    # Select only necessary columns
    safe_bets = safe_bets[["Player", "team", f"{target}", "Safe Bet", f"confidence_level_{target}"]]
    
    return safe_bets

# Example Usage:
safe_bets_df = get_safe_bets(results_ast,"AST" ,confidence_threshold=60)
print(safe_bets_df)

In [ ]:
def get_safe_bets(df,target ,confidence_threshold=60):
    """
    Filters the DataFrame for safe bets based on confidence threshold.
    Selects the lowest value in the stat range for a conservative bet.
    """
    safe_bets = df[df[f"confidence_level_{target}"] >= confidence_threshold].copy()
    
    # Extract the lowest stat value from the "Stat Range" column
    safe_bets["Safe Bet"] = safe_bets[target].apply(lambda x: int(x.split(" - ")[0]))
    
    # Select only necessary columns
    safe_bets = safe_bets[["Player", "team", f"{target}", "Safe Bet", f"confidence_level_{target}"]]
    
    return safe_bets

# Example Usage:
safe_bets_df = get_safe_bets(results_3pm,"3PM" ,confidence_threshold=50)
print(safe_bets_df)

In [ ]:
results_pts.head(10)

In [ ]:
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players

# Retrieve player ID (e.g., LeBron James)
for basketball_players in results_pts["Player"]:
    a_player = players.find_players_by_full_name(basketball_players)[0]
    player_id = a_player['id']

    gamelog = playergamelog.PlayerGameLog(player_id=player_id, season='2024-25')  # Use the latest season
    games = gamelog.get_data_frames()[0]  # Extract data frame
    
    latest_game = games.iloc[0]
    print(basketball_players,latest_game['PTS'])




# player_name = "LeBron James"
# player = players.find_players_by_full_name(player_name)[0]  # Get player data by name

# player_id = player['id']  # Get player ID

# # Fetch the player's game log (latest games)
# gamelog = playergamelog.PlayerGameLog(player_id=player_id, season='2024-25')  # Use the latest season
# games = gamelog.get_data_frames()[0]  # Extract data frame

# # Get the latest game data
# latest_game = games.iloc[0]  # Latest game data
# print(type(latest_game))
# print(latest_game["PTS"])
# display(latest_game)



In [ ]:
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.static import teams
import pandas as pd
from unidecode import unidecode

team_abbreviations = [
    "BOS", "BKN", "NYK", "PHI", "TOR",  # Atlantic Division
    "CHI", "CLE", "DET", "IND", "MIL",  # Central Division
    "ATL", "CHA", "MIA", "ORL", "WAS",  # Southeast Division
    "DEN", "MIN", "OKC", "POR", "UTA",  # Northwest Division
    "GSW", "LAC", "LAL", "PHX", "SAC",  # Pacific Division
    "DAL", "HOU", "MEM", "NOP", "SAS"   # Southwest Division
]


# Get all NBA teams and create a mapping of abbreviations to team IDs
nba_teams = teams.get_teams()
team_id_map = {team['abbreviation']: team['id'] for team in nba_teams}

# Dictionary to store team rosters
team_rosters = {}

# Loop through each team
for team_abbr in team_abbreviations:
    team_id = team_id_map[team_abbr]

    # Specify the season in 'YYYY-YY' format
    season = '2024-25'

    # Retrieve the team's roster
    roster = commonteamroster.CommonTeamRoster(team_id=team_id, season=season)

    # Convert the roster data to a pandas DataFrame
    roster_df = roster.get_data_frames()[0]

    # Normalize player names by removing accents
    roster_df['PLAYER_uni'] = roster_df['PLAYER'].apply(unidecode)


    roster_df['api_team_name'] = team_abbr

    # Store the roster for the team
    team_rosters[team_abbr] = roster_df





In [ ]:
# Display rosters
import pandas as pd
import os
for team_abbr, roster_df in team_rosters.items():
    print(f"\n{team_abbr} Roster:")
    display(roster_df[['PLAYER_uni','POSITION','api_team_name']])
    # roster = roster.DataFrame(ri)
    directory = "D:/roster_folder/2024"
    os.makedirs(directory, exist_ok=True)
    roster_df.to_csv(fr"{directory}/{team_abbr}_roster_file.csv", index=False)


In [ ]:


matched_players_list = []
for team_abbr, roster_df in team_rosters.items():
    # print(f"\n{team_abbr} Roster:")
    df_roster = roster_df[['PLAYER', 'PLAYER_uni']]

    if team_abbr in results_pts['team'].values:
        matched_players = df_roster[df_roster['PLAYER_uni'].isin(results_pts['Player'])]
        
        if not matched_players.empty:
            matched_players = matched_players['PLAYER']
            matched_players_list.append(matched_players)
            # display(matched_players)
            # print(matched_players)

# Combine all matched players into a single DataFrame
if matched_players_list:
    final_matched_df = pd.concat(matched_players_list, ignore_index=True)
    display(final_matched_df)  # Display the final combined DataFrame
else:
    print("No matches found.")


from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players

# Retrieve player ID (e.g., LeBron James)
player_list = final_matched_df.tolist()
for basketball_players in player_list:
    found_players = players.find_players_by_full_name(basketball_players)

    if found_players:
        a_player = found_players[0]
        player_id = a_player['id']

        # Fetch game log
        gamelog = playergamelog.PlayerGameLog(player_id=player_id, season='2024-25')
        games = gamelog.get_data_frames()[0]  # Extract DataFrame
        
        if not games.empty:
            latest_game = games.iloc[0]  # Get the most recent game
            print(f"{basketball_players}: {latest_game['PTS']} points")
        else:
            print(f"{basketball_players}: No game data available")

    else:
        print(f"Player not found: {basketball_players}")

In [ ]:

# display(results_reb)
# display(results_ast)
# display(results_pts)
# display(results_3pm)

# results_3pm = results_3pm[results_3pm["confidence_level_3PM"] != 100.00]
# df_merged = results_reb.merge(results_ast, on=['Player', 'team'], suffixes=('_reb', '_ast')) \
#     .merge(results_pts, on=['Player', 'team'], suffixes=('_pts','extra_pts')) \
#     .merge(results_3pm, on=['Player', 'team'], suffixes=('_pts','_3pm'))

# df_merged = df_merged[df_merged["Player"] != 'Tyrese Haliburton']


# print(df_merged.columns)


# df_merged['middlebet_PTS+REB'] = 0000

# df_merged['PTS+REB'] = df_merged[['middlebet_PTS', 'middlebet_REB']].sum(axis=1)
# df_merged['confidence_level_PTS+REB']=  round(df_merged[['confidence_level_PTS','confidence_level_REB']].sum(axis=1)/2,2)
# df_merged = df_merged.rename(columns={'FGA': 'PTS'})
# display(df_merged)
# display(df_merged[['Player','confidence_level_PTS+REB']])


from parlay_picker import create_parlays, create_parlays_high

num_groups = 10  # Number of parlays you want to generate
players_per_group = 9  # Number of players in each parlay
min_confidence = 70

parlays_df, group_confidence_scores = create_parlays_high(df_merged, num_groups, players_per_group,min_confidence, 70)


# Display the confidence scores for each group
print("Confidence scores for each group:")
for group, score in group_confidence_scores:
    print(f"{group}: {score:.2f}")


# Display the DataFrames for each parlay and their confidence scores
for (parlay_name, score), parlay_df in zip(group_confidence_scores, parlays_df):
    print("\n")
    print(f"{parlay_name}: total confidence score: {score:.2f}")
    display(parlay_df)
    print("\n")


In [ ]:
# Gets you player names
import pandas as pd

target = ["BOS","MEM", "CHI", "OKC", "BKN", "DAL", "HOU", "LAL"]
df_list = []  # List to store dataframes
for team in target:
    roster_path = f"D:\\roster_folder\\2024\\{team}_roster_file.csv"  # Construct the file path



    # Iterate through all files in the directory
    if os.path.exists(roster_path):  # Check if file exists
        df = pd.read_csv(roster_path)  # Read CSV file
        df_list.append(df)  # Append DataFrame to the list




        # Concatenate all the box score dataframes into one
        df= pd.concat(df_list, ignore_index=True)
        


        df =df['PLAYER_uni'].tolist()

list_of_names = df
print(list_of_names)

In [ ]:
# checks if player from roster is in history
import os

target = ["BOS","MEM", "CHI", "OKC", "BKN", "DAL", "HOU", "LAL"]
df_list = []  # List to store dataframes
for team in target:
    roster_path = f"D:\\roster_folder\\2024\\{team}_roster_file.csv"  # Construct the file path



    # Iterate through all files in the directory
    if os.path.exists(roster_path):  # Check if file exists
        df = pd.read_csv(roster_path)  # Read CSV file
        df_list.append(df)  # Append DataFrame to the list




        # Concatenate all the box score dataframes into one
        df= pd.concat(df_list, ignore_index=True)
        


        df =df['PLAYER_uni'].tolist()


list_of_names = df




unavaliable_names = []
for name in list_of_names:
    file_path = f"D:/nba_player_historic/nba_html_2023-24/{name}_content.html"
    # print(file_path)

    if os.path.exists(file_path):
        print("File exists! ")
    else:
        print("File does not exist.")
        unavaliable_names.append(name)

print(unavaliable_names)


In [ ]:
# this gets player and their team and put into a dictionary 
import os
import pandas as pd

# List of target teams
target = ["BOS", "MEM", "CHI", "OKC", "BKN", "DAL", "HOU", "LAL"]
df_list = []  # List to store DataFrames

# Loop through each team's roster file
for team in target:
    roster_path = f"D:\\roster_folder\\2024\\{team}_roster_file.csv"  # Construct the file path

    if os.path.exists(roster_path):  # Check if file exists
        df = pd.read_csv(roster_path)  # Read CSV file
        df_list.append(df)  # Store DataFrame in list
    else:
        print(f"File not found: {roster_path}")  # Debugging message

# Combine all DataFrames if any were found
if df_list:
    df_combined = pd.concat(df_list, ignore_index=True)

    # Extract two columns as a dictionary
    player_dict = dict(zip(df_combined['PLAYER_uni'], df_combined['api_team_name']))

    print(player_dict)  # Display dictionary
else:
    print("No valid roster files found.")
